In [16]:
import pandas as pd
import os

def compile_solar_with_temp(solar_folder, temp_csv, output_file):
    # Use os.getcwd() instead of __file__ for notebook compatibility
    base_path = os.getcwd() 
    folder_path = os.path.join(base_path, solar_folder)
    
    all_solar_data = []

    # 1. Process Solar XLS files
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith((".xls", ".xlsx")):
                file_path = os.path.join(folder_path, filename)
                try:
                    engine = 'xlrd' if filename.lower().endswith('.xls') else 'openpyxl'
                    df_raw = pd.read_excel(file_path, header=None, engine=engine).fillna('')
                    
                    header_row = None
                    for i, row in df_raw.iterrows():
                        row_vals = [str(val) for val in row.values]
                        if any("Date" in s for s in row_vals) and any("Generation" in s for s in row_vals):
                            header_row = i
                            break
                    
                    if header_row is not None:
                        df = pd.read_excel(file_path, header=header_row, engine=engine)
                        df.columns = df.columns.str.strip()
                        date_col = [c for c in df.columns if 'Date' in c][0]
                        gen_col = [c for c in df.columns if 'Generation' in c][0]
                        
                        temp_df = df[[date_col, gen_col]].copy()
                        temp_df.columns = ['Date', 'Generation']
                        temp_df['Date'] = pd.to_datetime(temp_df['Date'], errors='coerce')
                        all_solar_data.append(temp_df.dropna(subset=['Date']))
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

    if not all_solar_data:
        print("No solar data found.")
        return

    production_df = pd.concat(all_solar_data, ignore_index=True).sort_values(by='Date')

    # 2. Process Temperature & Solar CSV
    try:
        temp_path = os.path.join(base_path, temp_csv)
        with open(temp_path, 'r') as f:
            lines = f.readlines()
            
        csv_header_idx = None
        for i, line in enumerate(lines):
            if line.startswith("YEAR,MO,DY"):
                csv_header_idx = i
                break
        
        if csv_header_idx is not None:
            weather_df = pd.read_csv(temp_path, skiprows=csv_header_idx)
            weather_df['Date'] = pd.to_datetime(dict(year=weather_df['YEAR'], 
                                                     month=weather_df['MO'], 
                                                     day=weather_df['DY']))
            
            weather_df = weather_df[['Date', 'ALLSKY_SFC_SW_DWN', 'T2M_MAX', 'PRECTOTCORR']]
            weather_df.columns = ['Date', 'Solar_Irradiance', 'Max_Temp', 'Precipitation']
            
            final_df = pd.merge(production_df, weather_df, on='Date', how='left')
        else:
            final_df = production_df
            
    except Exception as e:
        print(f"Error processing CSV: {e}")
        final_df = production_df

    # 3. Filtering: Remove 0 generation and missing temperatures
    final_df = final_df[final_df['Generation'] > 0]
    if 'Max_Temp' in final_df.columns:
        # Filter out the -999 missing data markers and NaNs
        final_df = final_df[final_df['Max_Temp'] > -100]
        final_df = final_df.dropna(subset=['Max_Temp'])
        
    if 'Precipitation' in final_df.columns:
        final_df = final_df[final_df['Precipitation'] > -100]
        final_df = final_df.dropna(subset=['Precipitation'])

    # 4. Final Formatting: Convert to Month
    final_df = final_df.sort_values(by='Date')
    final_df['Date'] = final_df['Date'].dt.strftime('%B')

    # 5. Save as CSV (as requested previously)
    output_csv = output_file.replace('.xlsx', '.csv')
    final_df.to_csv(output_csv, index=False)
    
    print(f"\nSUCCESS: Created {output_csv} with {len(final_df)} rows.")

# Run the function
compile_solar_with_temp("solar_data", "max_temp_solar_prec.csv", "production.csv")


SUCCESS: Created production.csv with 359 rows.


In [17]:
import pandas as pd

def finalize_for_regression(input_file, output_file):
    # Load your current production data
    df = pd.read_csv(input_file)
    
    # Mapping for numerical conversion
    month_map = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    
    # 1. Create the numerical Month column
    df['Month'] = df['Date'].map(month_map)
    
    # 2. Remove the original string 'Date' column and any duplicates
    cols_to_remove = ['Date', 'Month_Num']
    df = df.drop(columns=[c for c in cols_to_remove if c in df.columns])
    
    # 3. Reorder columns so 'Month' is the first column
    cols = ['Month'] + [c for c in df.columns if c != 'Month']
    df = df[cols]
    
    # Save the final version
    df.to_csv(output_file, index=False)
    print(f"File saved as {output_file}")
    print(df.head())

# Run the update
finalize_for_regression('production.csv', 'production.csv')

File saved as production.csv
   Month  Generation  Solar_Irradiance  Max_Temp  Precipitation
0      5        11.5            3.8359     32.87           1.31
1      5        62.0            5.8344     38.13           0.00
2      5        54.7            5.2848     36.11           0.55
3      5        31.7            4.0356     36.17           4.41
4      5        50.9            5.8217     35.35           7.04


In [19]:
import pandas as pd

# 1. Load the data
df = pd.read_csv('production.csv')

# 2. Filter out the -999 missing data markers
# This is a critical step for linear regression
df_clean = df[df['Max_Temp'] > -100].copy()

# 3. Shuffle the data
# frac=1 means take 100% of the data, random_state ensures you can repeat the result
df_shuffled = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Save the ready-to-train file
df_shuffled.to_csv('production.csv', index=False)

print(f"Original rows: {len(df)}")
print(f"Cleaned rows (removed -999s): {len(df_clean)}")
print("Data has been shuffled and saved to 'production.csv'")

Original rows: 357
Cleaned rows (removed -999s): 357
Data has been shuffled and saved to 'production.csv'


**NOW WE TRAIN**

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Load your processed data
df = pd.read_csv('production.csv')

features = ['Month', 'Max_Temp', 'Solar_Irradiance', 'Precipitation'] 

X = df[features]
y = df['Generation']

# 4. Split data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Initialize and Train
model = LinearRegression()
model.fit(X_train, y_train)

# 6. Predict and Evaluate
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Model R-squared: {r2:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")
print(f"Mean Squared Error: {mse:.4f}")
print("Feature Weights (Coefficients):")
for feature, coef in zip(features, model.coef_):
    print(f"- {feature}: {coef:.4f}")

Model R-squared: 0.5621
Mean Absolute Error: 6.9830
Mean Squared Error: 91.7695
Feature Weights (Coefficients):
- Month: -0.1683
- Max_Temp: 0.0101
- Solar_Irradiance: 9.4677
- Precipitation: 0.1102


In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Load your data
df = pd.read_csv('production.csv')

# 2. Feature Engineering (Cyclical Months)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)

features = ['Month_Cos', 'Max_Temp', 'Solar_Irradiance', 'Precipitation'] 

X = df[features]
y = df['Generation']

# 5. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Train
model = LinearRegression()
model.fit(X_train, y_train)

# 7. Evaluate
predictions = model.predict(X_test)
print(f"Model R-squared: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.4f}")
print(f"Intercept: {model.intercept_:.4f}")
print("\nFeature Weights (Coefficients) for ESP32:")
for feature, coef in zip(features, model.coef_):
    print(f"- {feature}: {coef:.4f}")

Model R-squared: 0.5953
Mean Absolute Error: 6.7568
Intercept: 9.0488

Feature Weights (Coefficients) for ESP32:
- Month_Cos: -4.6126
- Max_Temp: -0.3670
- Solar_Irradiance: 9.5595
- Precipitation: 0.0235
